In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")
from databricks_connector import get_table

target_col = "btts"
odds_col: str = "goalNoGoal_quote_currentGG"


features = [
 'goalNoGoal_chance_goal',
 'goalNoGoal_chance_goalHome',
 'goalNoGoal_chance_goalAway',
 'goalNoGoal_multigoal_m13',
 'goalNoGoal_multigoal_m14',
 'goalNoGoal_multigoal_m24',
 'goalNoGoal_multigoal_m13Home',
 'goalNoGoal_multigoal_m13Away',
 'goalNoGoal_multigoal_m24Home',
 'goalNoGoal_multigoal_m24Away',
 'goalNoGoal_quote_realGG',
 'goalNoGoal_quote_initialGG',
 'goalNoGoal_quote_initialNG',
 'goalNoGoal_quote_currentGG',
 'goalNoGoal_quote_currentNG',
 'goalNoGoal_quote_diffRealCurrGG',
 'goalNoGoal_quote_diffRealCurrNG',
 'goalNoGoal_quote_diffInitialCurrGG',
 'goalNoGoal_quote_diffInitialCurrNG',
 'goalNoGoal_comparison_affini',
 'goalNoGoal_comparison_flashback',
 'goalNoGoal_stats_avgGoalHome',
 'goalNoGoal_stats_avgGoalTakenHome',
 'goalNoGoal_stats_avgGoalAway',
 'goalNoGoal_stats_avgGoalTakenAway',
 'goalNoGoal_flashback_goal',
 'goalNoGoal_flashback_m13',
 'goalNoGoal_flashback_m24',
 'goalNoGoal_flashback_m35',
 'underOver_chance_over05HT',
 'underOver_chance_over052HT',
 'underOver_chance_over15HT',
 'underOver_chance_over15',
 'underOver_chance_over25',
 'underOver_chance_over35',
 'underOver_chance_over45',
 'underOver_quote_realO',
 'underOver_quote_initialU',
 'underOver_quote_initialO',
 'underOver_quote_currentU',
 'underOver_quote_currentO',
 'underOver_quote_diffRealCurrU',
 'underOver_quote_diffRealCurrO',
 'underOver_quote_diffInitialCurrU',
 'underOver_quote_diffInitialCurrO',
 'underOver_comparison_affini',
 'underOver_comparison_flashback',
 'underOver_flashback_under05HT',
 'underOver_flashback_over05HT',
 'underOver_flashback_under15',
 'underOver_flashback_over15',
 'underOver_flashback_under25',
 'underOver_flashback_over25',
 'underOver_flashback_under35',
 'underOver_flashback_over35',
 'evaluation_valScala',
 'evaluation_valMetrica',
 'chance1x2_quote_current1',
 'chance1x2_quote_current2'
 ]

In [2]:
# Load data
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """
df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()

In [3]:
def time_train_val_split(
    df,
    time_col="time",
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    sort=True
):
    """
    Split temporale ordinato per train/val/test.

    Parametri
    ----------
    df : pd.DataFrame
    time_col : str
        Nome della colonna temporale.
    train_size, val_size, test_size : float
        Devono sommare a 1.0
    sort : bool
        Se True ordina per time_col crescente.

    Ritorna
    -------
    train_df, val_df, test_df
    """

    if round(train_size + val_size + test_size, 10) != 1.0:
        raise ValueError("train_size + val_size + test_size deve fare 1.0")

    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], utc=True, errors="coerce")

    out = out.dropna(subset=[time_col])

    if sort:
        out = out.sort_values(time_col).reset_index(drop=True)

    n = len(out)
    train_end = int(n * train_size)
    val_end = train_end + int(n * val_size)

    train_df = out.iloc[:train_end].copy()
    val_df = out.iloc[train_end:val_end].copy()
    test_df = out.iloc[val_end:].copy()

    return train_df, val_df, test_df

In [4]:
import numpy as np
import pandas as pd
import optuna


def build_optuna_rule_search_v3(
    train_df,
    val_df,
    features,
    target_col="y",
    odds_col="quota",
    continuous_max_unique=20,
    max_discrete_values=20,
    min_train_bets=80,
    min_val_bets=40,
    max_active_features=4,
    complexity_penalty=0.01,
    q_low=0.05,
    q_high=0.95,
):
    """
    Search di regole multivariate con Optuna.

    Continue:
      - ge  : x >= t
      - le  : x <= t
      - between : a <= x <= b

    Discrete:
      - eq : x == value

    Objective:
      score = min(ev_train, ev_val) * sqrt(min(n_train, n_val)) - penalty * n_rules
      con vincoli su n_train, n_val, ev_train, ev_val
    """

    train = train_df.copy()
    val = val_df.copy()

    train["return"] = np.where(train[target_col] == 1, train[odds_col] - 1.0, -1.0)
    val["return"] = np.where(val[target_col] == 1, val[odds_col] - 1.0, -1.0)

    feature_space = {}

    for feat in features:
        if feat not in train.columns or feat not in val.columns:
            continue

        s = train[feat].dropna()
        if s.empty:
            continue

        nunique = s.nunique()

        if pd.api.types.is_numeric_dtype(train[feat]) and nunique > continuous_max_unique:
            lo = float(s.quantile(q_low))
            hi = float(s.quantile(q_high))

            if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
                continue

            feature_space[feat] = {
                "kind": "continuous",
                "col": feat,
                "low": lo,
                "high": hi,
            }
        else:
            vals = train[feat].dropna().value_counts().index.tolist()
            vals = vals[:max_discrete_values]

            if len(vals) == 0:
                continue

            feature_space[feat] = {
                "kind": "discrete",
                "col": feat,
                "choices": vals,
            }

    def apply_rules_internal(df, rules):
        mask = pd.Series(True, index=df.index)

        for r in rules:
            col = r["col"]
            mode = r["mode"]

            if mode == "eq":
                mask &= df[col] == r["value"]
            elif mode == "ge":
                mask &= df[col] >= r["value"]
            elif mode == "le":
                mask &= df[col] <= r["value"]
            elif mode == "between":
                mask &= (df[col] >= r["low"]) & (df[col] <= r["high"])
            else:
                return df.iloc[0:0].copy()

        return df[mask].copy()

    def compute_score(tr_sub, va_sub, n_rules):
        n_train = len(tr_sub)
        n_val = len(va_sub)

        if n_train < min_train_bets or n_val < min_val_bets:
            return -1e9

        ev_train = tr_sub["return"].mean()
        ev_val = va_sub["return"].mean()

        if pd.isna(ev_train) or pd.isna(ev_val):
            return -1e9

        if ev_train <= 0 or ev_val <= 0:
            return -1e9

        score = (
            min(ev_train, ev_val) * np.sqrt(min(n_train, n_val))
            - complexity_penalty * n_rules
        )
        return float(score)

    def objective(trial):
        rules = []
        active_features = 0

        for feat, spec in feature_space.items():
            use_feat = trial.suggest_categorical(f"use__{feat}", [0, 1])

            if use_feat != 1:
                continue

            active_features += 1
            if active_features > max_active_features:
                return -1e9

            if spec["kind"] == "discrete":
                value = trial.suggest_categorical(f"value__{feat}", spec["choices"])
                rules.append({
                    "feature": feat,
                    "kind": "discrete",
                    "col": spec["col"],
                    "mode": "eq",
                    "value": value,
                })

            elif spec["kind"] == "continuous":
                mode = trial.suggest_categorical(f"mode__{feat}", ["ge", "le", "between"])
                lo = spec["low"]
                hi = spec["high"]

                if mode == "ge":
                    t = trial.suggest_float(f"thr__{feat}", lo, hi)
                    rules.append({
                        "feature": feat,
                        "kind": "continuous",
                        "col": spec["col"],
                        "mode": "ge",
                        "value": float(t),
                    })

                elif mode == "le":
                    t = trial.suggest_float(f"thr__{feat}", lo, hi)
                    rules.append({
                        "feature": feat,
                        "kind": "continuous",
                        "col": spec["col"],
                        "mode": "le",
                        "value": float(t),
                    })

                elif mode == "between":
                    a = trial.suggest_float(f"low__{feat}", lo, hi)
                    b = trial.suggest_float(f"high__{feat}", lo, hi)

                    if a > b:
                        a, b = b, a

                    rules.append({
                        "feature": feat,
                        "kind": "continuous",
                        "col": spec["col"],
                        "mode": "between",
                        "low": float(a),
                        "high": float(b),
                    })

        if len(rules) == 0:
            return -1e9

        tr_sub = apply_rules_internal(train, rules)
        va_sub = apply_rules_internal(val, rules)

        return compute_score(tr_sub, va_sub, len(rules))

    return train, val, feature_space, objective


def extract_best_rule_v3(study, feature_space):
    params = study.best_trial.params
    rules = []

    for feat, spec in feature_space.items():
        use_flag = params.get(f"use__{feat}", 0)
        if use_flag != 1:
            continue

        if spec["kind"] == "discrete":
            value = params.get(f"value__{feat}")
            if value is None:
                continue

            rules.append({
                "feature": feat,
                "kind": "discrete",
                "col": spec["col"],
                "mode": "eq",
                "value": value,
            })

        elif spec["kind"] == "continuous":
            mode = params.get(f"mode__{feat}")
            if mode is None:
                continue

            if mode in ["ge", "le"]:
                thr = params.get(f"thr__{feat}")
                if thr is None:
                    continue

                rules.append({
                    "feature": feat,
                    "kind": "continuous",
                    "col": spec["col"],
                    "mode": mode,
                    "value": float(thr),
                })

            elif mode == "between":
                a = params.get(f"low__{feat}")
                b = params.get(f"high__{feat}")
                if a is None or b is None:
                    continue

                a = float(a)
                b = float(b)
                if a > b:
                    a, b = b, a

                rules.append({
                    "feature": feat,
                    "kind": "continuous",
                    "col": spec["col"],
                    "mode": "between",
                    "low": a,
                    "high": b,
                })

    return rules


def apply_rules_v3(df, rules, target_col="y", odds_col="quota"):
    out = df.copy()
    out["return"] = np.where(out[target_col] == 1, out[odds_col] - 1.0, -1.0)

    mask = pd.Series(True, index=out.index)

    for r in rules:
        if r["mode"] == "eq":
            mask &= out[r["col"]] == r["value"]
        elif r["mode"] == "ge":
            mask &= out[r["col"]] >= r["value"]
        elif r["mode"] == "le":
            mask &= out[r["col"]] <= r["value"]
        elif r["mode"] == "between":
            mask &= (out[r["col"]] >= r["low"]) & (out[r["col"]] <= r["high"])

    sel = out[mask].copy()

    stats = {
        "n_bets": int(len(sel)),
        "win_rate": float(sel[target_col].mean()) if len(sel) > 0 else np.nan,
        "ev": float(sel["return"].mean()) if len(sel) > 0 else np.nan,
        "profit": float(sel["return"].sum()) if len(sel) > 0 else 0.0,
        "quota_media": float(sel[odds_col].mean()) if len(sel) > 0 else np.nan,
    }

    return sel, stats


def pretty_rule_v3(rules):
    parts = []

    for r in rules:
        feat = r["feature"]

        if r["mode"] == "eq":
            parts.append(f"{feat} == {repr(r['value'])}")
        elif r["mode"] == "ge":
            parts.append(f"{feat} >= {r['value']:.6g}")
        elif r["mode"] == "le":
            parts.append(f"{feat} <= {r['value']:.6g}")
        elif r["mode"] == "between":
            parts.append(f"{r['low']:.6g} <= {feat} <= {r['high']:.6g}")

    return " AND ".join(parts)

In [5]:
train_df, val_df, test_df = time_train_val_split(
    df,
    time_col="time",
    train_size=0.6,
    val_size=0.2,
    test_size=0.2
)

train_prep, val_prep, feature_space, objective = build_optuna_rule_search_v3(
    train_df=train_df,
    val_df=val_df,
    features=features,
    target_col=target_col,
    odds_col=odds_col,
    min_train_bets=80,
    min_val_bets=40,
    max_active_features=4,
    complexity_penalty=0.01,
    q_low=0.05,
    q_high=0.95,
)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=500)

rules = extract_best_rule_v3(study, feature_space)

print(rules)
print(pretty_rule_v3(rules))

val_sel, val_stats = apply_rules_v3(
    val_prep,
    rules,
    target_col=target_col,
    odds_col=odds_col
)
print(val_stats)

[I 2026-03-15 17:34:37,515] A new study created in memory with name: no-name-fce5ce83-e7be-43c0-8bba-55fc77ec74bc
[I 2026-03-15 17:34:37,517] Trial 0 finished with value: -1000000000.0 and parameters: {'use__goalNoGoal_chance_goal': 0, 'use__goalNoGoal_chance_goalHome': 1, 'mode__goalNoGoal_chance_goalHome': 'between', 'low__goalNoGoal_chance_goalHome': 80.47785247694208, 'high__goalNoGoal_chance_goalHome': 93.04227458757705, 'use__goalNoGoal_chance_goalAway': 1, 'mode__goalNoGoal_chance_goalAway': 'le', 'thr__goalNoGoal_chance_goalAway': 71.54152171427306, 'use__goalNoGoal_multigoal_m13': 0, 'use__goalNoGoal_multigoal_m14': 0, 'use__goalNoGoal_multigoal_m24': 0, 'use__goalNoGoal_multigoal_m13Home': 0, 'use__goalNoGoal_multigoal_m13Away': 1, 'mode__goalNoGoal_multigoal_m13Away': 'ge', 'thr__goalNoGoal_multigoal_m13Away': 72.41245165704643, 'use__goalNoGoal_multigoal_m24Home': 1, 'mode__goalNoGoal_multigoal_m24Home': 'between', 'low__goalNoGoal_multigoal_m24Home': 54.40160657954894, 'hi

[{'feature': 'goalNoGoal_chance_goalHome', 'kind': 'continuous', 'col': 'goalNoGoal_chance_goalHome', 'mode': 'between', 'low': 80.47785247694208, 'high': 93.04227458757705}, {'feature': 'goalNoGoal_chance_goalAway', 'kind': 'continuous', 'col': 'goalNoGoal_chance_goalAway', 'mode': 'le', 'value': 71.54152171427306}, {'feature': 'goalNoGoal_multigoal_m13Away', 'kind': 'continuous', 'col': 'goalNoGoal_multigoal_m13Away', 'mode': 'ge', 'value': 72.41245165704643}, {'feature': 'goalNoGoal_multigoal_m24Home', 'kind': 'continuous', 'col': 'goalNoGoal_multigoal_m24Home', 'mode': 'between', 'low': 37.41781912908543, 'high': 54.40160657954894}]
80.4779 <= goalNoGoal_chance_goalHome <= 93.0423 AND goalNoGoal_chance_goalAway <= 71.5415 AND goalNoGoal_multigoal_m13Away >= 72.4125 AND 37.4178 <= goalNoGoal_multigoal_m24Home <= 54.4016
{'n_bets': 6, 'win_rate': 0.5, 'ev': -0.11166666666666669, 'profit': -0.6700000000000002, 'quota_media': 1.7966666666666666}


In [6]:
test_sel, test_stats = apply_rules_v3(
    test_df,
    rules,
    target_col=target_col,
    odds_col=odds_col
)
print(test_stats)

{'n_bets': 13, 'win_rate': 0.6153846153846154, 'ev': 0.06615384615384616, 'profit': 0.8600000000000001, 'quota_media': 1.7546153846153847}
